In [37]:
import json
import os
from pathlib import Path
import pandas as pd

In [38]:
prompt_dir = Path('../reports/llm_reasoning/prompts')
evaluation_dir = Path('../reports/evaluation')

prompts = {
    'TP': (prompt_dir / 'tp_prompt.txt').read_text(),
    'FN': (prompt_dir / 'fn_prompt.txt').read_text(),
    'FP': (prompt_dir / 'fp_prompt.txt').read_text(),
    'TN': (prompt_dir / 'tn_prompt.txt').read_text()
}

rubric_df = pd.read_csv(
    evaluation_dir / 'explanation_evaluation_rubric.csv',
    index_col=0
)

with open(evaluation_dir / 'evaluation_weights.json') as f:
    evaluation_weights = json.load(f)

print(prompts.keys())
print(rubric_df.shape)
print(evaluation_weights)

dict_keys(['TP', 'FN', 'FP', 'TN'])
(5, 6)
{'faithfulness_no_hallucination': 0.3, 'clinical_plausibility': 0.25, 'caution_awareness': 0.2, 'completeness': 0.15, 'clarity': 0.1}


In [39]:
print(prompts['TP'][:1000])

You are a clinical AI explanation assistant.

Your task is to explain why the mortality prediction model made its prediction for this ICU patient.

Use only the provided evidence. Do not invent clinical facts. Do not use the true label to justify the prediction; use it only as case metadata. If a feature has a caution flag, mention it carefully.

Patient prediction:
- Case type: TP
- True label: 1
- Predicted label: 1
- Predicted mortality probability: 0.9995

Risk-increasing evidence:
[
  {
    "feature": "d1_heartrate_min",
    "value": 0.0,
    "shap_value": 2.0046439410543497,
    "direction": "risk_increasing",
    "clinical_meaning": "extreme or very low heart rate may indicate severe instability or data quality issue",
    "caution_flags": [
      "Zero-valued vital sign; may reflect extreme clinical event or recording artifact."
    ]
  },
  {
    "feature": "d1_spo2_min",
    "value": 31.0,
    "shap_value": 0.9487559313200912,
    "direction": "risk_increasing",
    "clinical

In [40]:
from dotenv import load_dotenv

load_dotenv()

openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print('OPENAI_API_KEY loaded.')
else:
    print('OPENAI_API_KEY not found.')

OPENAI_API_KEY loaded.


In [41]:
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

print('OpenAI client initialized.')

OpenAI client initialized.


In [42]:
GENERATOR_MODEL = 'gpt-4.1-mini'
EVALUATOR_MODEL = 'gpt-4o'

print(f'Generator model: {GENERATOR_MODEL}')
print(f'Evaluator model: {EVALUATOR_MODEL}')

Generator model: gpt-4.1-mini
Evaluator model: gpt-4o


In [43]:
test_response = client.responses.create(
    model=GENERATOR_MODEL,
    input='Reply with exactly: API connection works.'
)

print(test_response.output_text)

API connection works.


### Stricter Generator and Evaluator Prompts

The first generated explanation referenced the true outcome, and the evaluator did not penalize this strongly enough. Therefore, the generator and evaluator prompts were tightened before continuing.

In [44]:
generator_system_message = """
You are a clinical AI explanation generator.

Your role is to generate a concise, evidence-grounded explanation for an ICU mortality prediction model.

Hard rules:
- Use only the provided evidence.
- Do not invent clinical facts.
- Do not mention or use the true label in the explanation.
- Do not say the prediction was correct, incorrect, consistent with the true outcome, or inconsistent with the true outcome.
- Do not add measurement units unless they are explicitly present in the evidence.
- Do not invent normal ranges, diagnoses, mechanisms, or clinical details beyond the provided clinical_meaning.
- Correctly distinguish risk-increasing and risk-decreasing evidence.
- Mention caution flags when present.
- If a value has a caution flag, explain that it may require careful interpretation.
- Keep the explanation clinically grounded and concise.

Write the explanation using these sections:
1. Prediction summary
2. Main risk-increasing factors
3. Main risk-decreasing factors
4. Caution notes
5. Overall interpretation
"""

In [45]:
def build_evaluator_prompt(explanation, original_prompt, rubric_df, evaluation_weights):
    rubric_text = rubric_df.to_string()

    prompt = f"""
You are a strict explanation evaluation agent.

Your task is to evaluate a generated clinical explanation using the predefined rubric.

Important evaluation rules:
- Evaluate the explanation against the original evidence and prompt.
- Do not reward unsupported claims.
- Penalize hallucinated clinical facts.
- Penalize missing caution flags.
- Penalize invented units, normal ranges, diagnoses, or clinical details that are not explicitly present in the evidence.
- Penalize incorrect interpretation of SHAP direction.
- Penalize any use of the true label to justify or describe the prediction.
- If the explanation says the prediction is correct, consistent with the true outcome, or uses the true label as justification, faithfulness_no_hallucination must be at most 3.
- If the explanation adds measurement units not present in the evidence, faithfulness_no_hallucination should be reduced.
- If there is any hallucination, unsupported clinical claim, or true-label leakage, revision_needed must be true.
- - If faithfulness_no_hallucination is 3 or lower, revision_needed must be true.
- Use the rubric exactly as defined.

Original prompt and evidence:
{original_prompt}

Generated explanation:
{explanation}

Evaluation rubric:
{rubric_text}

Evaluation weights:
{json.dumps(evaluation_weights, indent=2)}

Return your evaluation in JSON format with this structure:

{{
  "scores": {{
    "faithfulness_no_hallucination": {{"score": 1-5, "rationale": "..."}},
    "clinical_plausibility": {{"score": 1-5, "rationale": "..."}},
    "caution_awareness": {{"score": 1-5, "rationale": "..."}},
    "completeness": {{"score": 1-5, "rationale": "..."}},
    "clarity": {{"score": 1-5, "rationale": "..."}}
  }},
  "overall_feedback": "...",
  "revision_needed": true/false,
  "revision_instructions": "..."
}}
"""
    return prompt.strip()

In [46]:
manual_explanation_path = Path('../reports/llm_reasoning/manual_explanations/tp_manual_explanation.txt')

tp_manual_explanation = manual_explanation_path.read_text()

print(tp_manual_explanation)


1. Prediction summary
The model assigned this patient a very high predicted probability of in-hospital mortality (0.9995).

2. Main risk-increasing factors
The prediction was mainly driven upward by severe physiological instability. The strongest risk-increasing evidence was d1_heartrate_min = 0, which had the largest positive SHAP contribution. This value should be interpreted cautiously because a zero-valued heart rate may represent either an extreme clinical event or a recording artifact. Other major risk-increasing factors included very low minimum oxygen saturation (d1_spo2_min = 31), mechanical ventilation, very low systolic blood pressure (d1_sysbp_min = 41), low GCS motor score, low bicarbonate values, and low platelet count.

3. Main risk-decreasing factors
A few features slightly decreased the predicted risk, including d1_wbc_min, d1_bun_max, hematocrit-related features, BMI, and height. However, these negative contributions were much smaller than the strong risk-increasing 

In [47]:
tp_evaluator_prompt = build_evaluator_prompt(
    explanation=tp_manual_explanation,
    original_prompt=prompts['TP'],
    rubric_df=rubric_df,
    evaluation_weights=evaluation_weights
)

print(tp_evaluator_prompt[:2000])

You are a strict explanation evaluation agent.

Your task is to evaluate a generated clinical explanation using the predefined rubric.

Important evaluation rules:
- Evaluate the explanation against the original evidence and prompt.
- Do not reward unsupported claims.
- Penalize hallucinated clinical facts.
- Penalize missing caution flags.
- Penalize invented units, normal ranges, diagnoses, or clinical details that are not explicitly present in the evidence.
- Penalize incorrect interpretation of SHAP direction.
- Penalize any use of the true label to justify or describe the prediction.
- If the explanation says the prediction is correct, consistent with the true outcome, or uses the true label as justification, faithfulness_no_hallucination must be at most 3.
- If the explanation adds measurement units not present in the evidence, faithfulness_no_hallucination should be reduced.
- If there is any hallucination, unsupported clinical claim, or true-label leakage, revision_needed must 

In [48]:
tp_generation_response = client.responses.create(
    model=GENERATOR_MODEL,
    instructions=generator_system_message,
    input=prompts['TP'],
    temperature=0.2
)

tp_generated_explanation = tp_generation_response.output_text

print(tp_generated_explanation)

1. Prediction summary  
The model predicted a very high mortality risk (probability 0.9995) for this ICU patient.

2. Main risk-increasing factors  
Key contributors to increased risk include an extremely low minimum heart rate (0.0), which may indicate severe instability or a data artifact; very low minimum oxygen saturation (31.0) indicating hypoxemia; mechanical ventilation reflecting critical respiratory failure; low bicarbonate levels (minimum 8.0, maximum 12.0) suggesting metabolic acidosis; very low minimum systolic blood pressure (41.0) indicating hemodynamic instability; impaired neurological response with a low GCS motor score (1.0); and low platelet count (58.0), which may reflect severe illness or coagulation abnormalities.

3. Main risk-decreasing factors  
Factors slightly reducing risk include a low white blood cell count (1.1), elevated BUN (18.0), and other parameters such as hematocrit, mean blood pressure, diastolic blood pressure, BMI, and height, though these have 

In [49]:
tp_generated_evaluator_prompt = build_evaluator_prompt(
    explanation=tp_generated_explanation,
    original_prompt=prompts['TP'],
    rubric_df=rubric_df,
    evaluation_weights=evaluation_weights
)

tp_eval_response = client.responses.create(
    model=EVALUATOR_MODEL,
    input=tp_generated_evaluator_prompt,
    temperature=0.0
)

tp_eval_text = tp_eval_response.output_text

print(tp_eval_text)

```json
{
  "scores": {
    "faithfulness_no_hallucination": {
      "score": 3,
      "rationale": "The explanation is mostly grounded in the evidence but includes minor unsupported phrasing. It accurately reflects SHAP directions but does not fully address the caution flag for the zero-valued heart rate."
    },
    "clinical_plausibility": {
      "score": 4,
      "rationale": "The explanation is clinically plausible and interprets most important signals clearly, though it could provide more depth in some areas."
    },
    "caution_awareness": {
      "score": 4,
      "rationale": "Correctly mentions the caution flag for the zero-valued heart rate but could integrate it more fully into the overall interpretation."
    },
    "completeness": {
      "score": 4,
      "rationale": "Covers most important risk-increasing and risk-decreasing factors with a clear overall interpretation, but could give more attention to the risk-decreasing evidence."
    },
    "clarity": {
      "score

### Revision Loop

Because the evaluator marked the generated explanation as requiring revision, the generator is asked to revise the explanation using the evaluator feedback while preserving evidence-grounding rules.

In [50]:
revision_prompt = f"""
The following clinical explanation was evaluated and marked as needing revision.

Original evidence and prompt:
{prompts['TP']}

Generated explanation:
{tp_generated_explanation}

Evaluator feedback:
{tp_eval_text}

Revise the explanation according to the evaluator feedback.

Rules:
- Use only the original evidence.
- Do not add new clinical facts.
- Do not mention the true label.
- Do not say whether the prediction was correct or incorrect.
- Do not add measurement units unless explicitly present in the evidence.
- Preserve the same section structure:
  1. Prediction summary
  2. Main risk-increasing factors
  3. Main risk-decreasing factors
  4. Caution notes
  5. Overall interpretation
"""

tp_revision_response = client.responses.create(
    model=GENERATOR_MODEL,
    instructions=generator_system_message,
    input=revision_prompt,
    temperature=0.2
)

tp_revised_explanation = tp_revision_response.output_text

print(tp_revised_explanation)

1. Prediction summary  
The model predicted a very high mortality risk (probability 0.9995) for this ICU patient.

2. Main risk-increasing factors  
Key contributors to increased risk include an extremely low minimum heart rate (0.0), which may indicate severe instability or a data artifact; very low minimum oxygen saturation (31.0) indicating hypoxemia; mechanical ventilation reflecting severe respiratory failure or critical illness; low bicarbonate levels (minimum 8.0, maximum 12.0) suggesting metabolic acidosis; very low minimum systolic blood pressure (41.0) indicating hemodynamic instability; impaired neurological response with a low GCS motor score (1.0); and low platelet count (58.0), which may reflect severe illness or coagulation abnormalities.

3. Main risk-decreasing factors  
Factors slightly reducing risk include a low white blood cell count (1.1), which reflects inflammatory or immune status, and an elevated BUN (18.0), which may indicate renal dysfunction or metabolic st

In [51]:
tp_revised_evaluator_prompt = build_evaluator_prompt(
    explanation=tp_revised_explanation,
    original_prompt=prompts['TP'],
    rubric_df=rubric_df,
    evaluation_weights=evaluation_weights
)

tp_revised_eval_response = client.responses.create(
    model=EVALUATOR_MODEL,
    input=tp_revised_evaluator_prompt,
    temperature=0.0
)

tp_revised_eval_text = tp_revised_eval_response.output_text

print(tp_revised_eval_text)

```json
{
  "scores": {
    "faithfulness_no_hallucination": {
      "score": 3,
      "rationale": "The explanation is mostly grounded in the evidence but includes minor unsupported phrasing, such as 'modest risk reduction from immune and renal-related markers,' which is not explicitly supported by the evidence. The SHAP directions are mostly reflected correctly."
    },
    "clinical_plausibility": {
      "score": 4,
      "rationale": "The explanation is clinically plausible and interprets most important signals clearly, such as the implications of low heart rate and oxygen saturation. However, it could provide more depth in some areas."
    },
    "caution_awareness": {
      "score": 5,
      "rationale": "Correctly mentions and explains the caution flag related to the zero-valued heart rate, integrating it appropriately into the interpretation."
    },
    "completeness": {
      "score": 4,
      "rationale": "Covers most important risk-increasing and risk-decreasing factors wi

### Stricter Second Revision

The first revision improved caution awareness but introduced unsupported clinical interpretations for some risk-decreasing features. A second stricter revision is performed to keep all statements directly tied to the provided evidence.

In [52]:
strict_revision_prompt = f"""
The revised explanation was evaluated and still marked as needing revision.

Original evidence and prompt:
{prompts['TP']}

Previous revised explanation:
{tp_revised_explanation}

Evaluator feedback:
{tp_revised_eval_text}

Create a stricter revised explanation.

Rules:
- Use only the original evidence.
- Do not add new clinical facts.
- Do not mention the true label.
- Do not say whether the prediction was correct or incorrect.
- Do not add measurement units unless explicitly present in the evidence.
- Do not infer clinical meanings for features unless the evidence explicitly provides a clinical_meaning.
- For risk-decreasing features without explicit clinical_meaning, only say that they reduced the model's predicted risk.
- Avoid phrases like renal dysfunction, immune status, inflammatory status, metabolic stress, coagulation abnormalities, or mitigating factors unless they are explicitly present in the evidence.
- Preserve the same section structure:
  1. Prediction summary
  2. Main risk-increasing factors
  3. Main risk-decreasing factors
  4. Caution notes
  5. Overall interpretation
"""

tp_strict_revision_response = client.responses.create(
    model=GENERATOR_MODEL,
    instructions=generator_system_message,
    input=strict_revision_prompt,
    temperature=0.1
)

tp_strict_revised_explanation = tp_strict_revision_response.output_text

print(tp_strict_revised_explanation)

1. Prediction summary  
The model predicted a very high mortality risk (probability 0.9995) for this ICU patient.

2. Main risk-increasing factors  
Key contributors to increased risk include an extremely low minimum heart rate (0.0), which may indicate severe instability or a data quality issue; very low minimum oxygen saturation (31.0) indicating hypoxemia; mechanical ventilation reflecting severe respiratory failure or critical illness; low bicarbonate levels (minimum 8.0, maximum 12.0) suggesting metabolic acidosis; very low minimum systolic blood pressure (41.0) indicating hemodynamic instability; low Glasgow Coma Scale motor score (1.0) indicating impaired neurological response; and low platelet count (58.0).

3. Main risk-decreasing factors  
Factors that reduced the predicted risk include low minimum white blood cell count (1.1), elevated maximum blood urea nitrogen (18.0), and several features without specific clinical interpretation provided: minimum hematocrit (19.5), maximu

In [53]:
tp_strict_revised_evaluator_prompt = build_evaluator_prompt(
    explanation=tp_strict_revised_explanation,
    original_prompt=prompts['TP'],
    rubric_df=rubric_df,
    evaluation_weights=evaluation_weights
)

tp_strict_revised_eval_response = client.responses.create(
    model=EVALUATOR_MODEL,
    input=tp_strict_revised_evaluator_prompt,
    temperature=0.0
)

tp_strict_revised_eval_text = tp_strict_revised_eval_response.output_text

print(tp_strict_revised_eval_text)

```json
{
  "scores": {
    "faithfulness_no_hallucination": {
      "score": 4,
      "rationale": "The explanation is mostly grounded in the evidence and accurately reflects SHAP directions. However, it uses the true label to justify the prediction, which is not allowed."
    },
    "clinical_plausibility": {
      "score": 5,
      "rationale": "The explanation is clinically strong, coherent, and appropriately cautious, interpreting the evidence in a reasonable way."
    },
    "caution_awareness": {
      "score": 5,
      "rationale": "Correctly mentions and explains the caution flag related to the zero-valued heart rate."
    },
    "completeness": {
      "score": 5,
      "rationale": "Comprehensively covers the main evidence, including risk-increasing factors, risk-decreasing factors, caution-relevant evidence, and overall interpretation."
    },
    "clarity": {
      "score": 5,
      "rationale": "The explanation is very clear, concise, and well structured."
    }
  },
  "o

In [54]:
forbidden_true_label_phrases = [
    'true outcome',
    'true label',
    'actual outcome',
    'actual label',
    'correct prediction',
    'incorrect prediction',
    'prediction was correct',
    'prediction was incorrect',
    'consistent with the true outcome',
    'consistent with the actual outcome'
]

def check_forbidden_phrases(text, phrases):
    text_lower = text.lower()
    found = [phrase for phrase in phrases if phrase in text_lower]
    return found

strict_revision_forbidden_phrases = check_forbidden_phrases(
    tp_strict_revised_explanation,
    forbidden_true_label_phrases
)

print('Forbidden true-label phrases found:')
print(strict_revision_forbidden_phrases)

Forbidden true-label phrases found:
[]


### Weighted Evaluation Scores

The evaluator returns separate rubric scores for each explanation quality dimension. To summarize these results, a weighted overall score is computed using the predefined evaluation weights. This score is used as a numeric summary, but it is interpreted together with evaluator feedback and rule-based checks.

In [55]:
import re

def extract_json_from_response(response_text):
    json_match = re.search(r'\{.*\}', response_text, re.DOTALL)

    if json_match is None:
        raise ValueError('No JSON object found in response text.')

    return json.loads(json_match.group(0))


def compute_weighted_score(evaluation_result, evaluation_weights):
    weighted_score = 0

    for metric, weight in evaluation_weights.items():
        score = evaluation_result['scores'][metric]['score']
        weighted_score += score * weight

    return weighted_score

In [56]:
tp_initial_eval_json = extract_json_from_response(tp_eval_text)
tp_revised_eval_json = extract_json_from_response(tp_revised_eval_text)
tp_strict_revised_eval_json = extract_json_from_response(tp_strict_revised_eval_text)

weighted_scores = {
    'initial_generated': compute_weighted_score(
        tp_initial_eval_json,
        evaluation_weights
    ),
    'first_revision': compute_weighted_score(
        tp_revised_eval_json,
        evaluation_weights
    ),
    'strict_second_revision': compute_weighted_score(
        tp_strict_revised_eval_json,
        evaluation_weights
    )
}

weighted_scores

{'initial_generated': 3.8000000000000003,
 'first_revision': 4.0,
 'strict_second_revision': 4.7}

In [57]:
weighted_scores_df = pd.DataFrame([
    {
        'version': version,
        'weighted_score': score,
        'revision_needed': {
            'initial_generated': tp_initial_eval_json['revision_needed'],
            'first_revision': tp_revised_eval_json['revision_needed'],
            'strict_second_revision': tp_strict_revised_eval_json['revision_needed']
        }[version],
        'forbidden_true_label_phrases': {
            'initial_generated': None,
            'first_revision': None,
            'strict_second_revision': strict_revision_forbidden_phrases
        }[version]
    }
    for version, score in weighted_scores.items()
])

weighted_scores_df

,version,weighted_score,revision_needed,forbidden_true_label_phrases
0,initial_generated,3.8,True,None
1,first_revision,4.0,True,None
2,strict_second_revision,4.7,True,[]


### Apply Final Generator-Evaluator Setup to FN, FP, and TN Cases

After refining the generator and evaluator prompts on the TP case, the final stricter setup is applied to the remaining case types: FN, FP, and TN. For these cases, the workflow uses generation, evaluation, weighted scoring, and rule-based forbidden phrase checks. Revision is only considered if major issues appear.

In [58]:
def generate_explanation(case_type):
    response = client.responses.create(
        model=GENERATOR_MODEL,
        instructions=generator_system_message,
        input=prompts[case_type],
        temperature=0.2
    )

    return response.output_text


def evaluate_explanation(explanation, case_type):
    evaluator_prompt = build_evaluator_prompt(
        explanation=explanation,
        original_prompt=prompts[case_type],
        rubric_df=rubric_df,
        evaluation_weights=evaluation_weights
    )

    response = client.responses.create(
        model=EVALUATOR_MODEL,
        input=evaluator_prompt,
        temperature=0.0
    )

    return response.output_text


def summarize_evaluation(evaluation_text, evaluation_weights):
    evaluation_json = extract_json_from_response(evaluation_text)

    return {
        'weighted_score': compute_weighted_score(
            evaluation_json,
            evaluation_weights
        ),
        'revision_needed': evaluation_json['revision_needed'],
        'faithfulness_no_hallucination': evaluation_json['scores']['faithfulness_no_hallucination']['score'],
        'clinical_plausibility': evaluation_json['scores']['clinical_plausibility']['score'],
        'caution_awareness': evaluation_json['scores']['caution_awareness']['score'],
        'completeness': evaluation_json['scores']['completeness']['score'],
        'clarity': evaluation_json['scores']['clarity']['score'],
        'overall_feedback': evaluation_json['overall_feedback'],
        'revision_instructions': evaluation_json['revision_instructions']
    }

In [60]:
def safe_summarize_evaluation(evaluation_text, evaluation_weights):
    try:
        evaluation_json = extract_json_from_response(evaluation_text)

        return {
            'weighted_score': compute_weighted_score(
                evaluation_json,
                evaluation_weights
            ),
            'revision_needed': evaluation_json['revision_needed'],
            'faithfulness_no_hallucination': evaluation_json['scores']['faithfulness_no_hallucination']['score'],
            'clinical_plausibility': evaluation_json['scores']['clinical_plausibility']['score'],
            'caution_awareness': evaluation_json['scores']['caution_awareness']['score'],
            'completeness': evaluation_json['scores']['completeness']['score'],
            'clarity': evaluation_json['scores']['clarity']['score'],
            'overall_feedback': evaluation_json.get('overall_feedback', ''),
            'revision_instructions': evaluation_json.get('revision_instructions', ''),
            'parse_error': None
        }

    except Exception as e:
        return {
            'weighted_score': None,
            'revision_needed': None,
            'faithfulness_no_hallucination': None,
            'clinical_plausibility': None,
            'caution_awareness': None,
            'completeness': None,
            'clarity': None,
            'overall_feedback': None,
            'revision_instructions': None,
            'parse_error': str(e)
        }

In [61]:
case_results = {}

for case_type in ['FN', 'FP', 'TN']:
    explanation = generate_explanation(case_type)
    evaluation_text = evaluate_explanation(explanation, case_type)
    evaluation_summary = safe_summarize_evaluation(
        evaluation_text,
        evaluation_weights
    )

    case_results[case_type] = {
        'explanation': explanation,
        'evaluation_text': evaluation_text,
        'evaluation_summary': evaluation_summary,
        'forbidden_true_label_phrases': check_forbidden_phrases(
            explanation,
            forbidden_true_label_phrases
        )
    }

    print(f'=== {case_type} Explanation ===')
    print(explanation)
    print('\n=== Evaluation ===')
    print(evaluation_text)
    print('\n=== Evaluation Summary ===')
    print(evaluation_summary)
    print('\n=== Forbidden phrases ===')
    print(case_results[case_type]['forbidden_true_label_phrases'])
    print('\n' + '=' * 80 + '\n')

=== FN Explanation ===
1. Prediction summary  
The model predicted a very low mortality probability (0.37%) for this ICU patient.

2. Main risk-increasing factors  
Older age (72 years) and a very high maximum respiratory rate on day 1 (56) increased mortality risk, as older age is linked to higher risk and elevated respiratory rate may indicate respiratory distress. Additional risk was contributed by the diagnosis category (apache_3j_diagnosis 111.01) and low minimum diastolic blood pressure on day 1 (27), though no specific clinical interpretation is provided for these. Elevated mean arterial pressure and maximum noninvasive mean blood pressure (both 166) also increased risk without predefined clinical meaning.

3. Main risk-decreasing factors  
Absence of mechanical ventilation strongly decreased risk, as ventilation indicates severe illness. A normal minimum white blood cell count (6) suggested lower inflammatory or immune activation. The ICU unit identifier was associated with low

In [62]:
all_case_summary_df = pd.DataFrame([
    {
        'case_type': case_type,
        **result['evaluation_summary'],
        'forbidden_true_label_phrases': result['forbidden_true_label_phrases']
    }
    for case_type, result in case_results.items()
])

all_case_summary_df

,case_type,weighted_score,revision_needed,faithfulness_no_hallucination,clinical_plausibility,caution_awareness,completeness,clarity,overall_feedback,revision_instructions,parse_error,forbidden_true_label_phrases
0,FN,3.8,True,3.0,4.0,4.0,4.0,5.0,The explanation is mostly grounded in the evid...,Remove unsupported claims about 'normal' value...,None,[]
1,FP,3.8,True,3.0,4.0,4.0,4.0,5.0,The explanation is mostly faithful to the evid...,Ensure all statements are fully supported by t...,None,[]
2,TN,NaN,None,NaN,NaN,NaN,NaN,NaN,None,None,"Expecting ',' delimiter: line 26 column 2 (cha...",[]


In [63]:
def evaluate_explanation_json_only(explanation, case_type):
    evaluator_prompt = build_evaluator_prompt(
        explanation=explanation,
        original_prompt=prompts[case_type],
        rubric_df=rubric_df,
        evaluation_weights=evaluation_weights
    )

    evaluator_prompt += """

Return only valid JSON.
Do not wrap the JSON in markdown.
Do not include explanations outside the JSON.
Make sure all braces and commas are valid.
"""

    response = client.responses.create(
        model=EVALUATOR_MODEL,
        input=evaluator_prompt,
        temperature=0.0
    )

    return response.output_text

In [64]:
tn_evaluation_text_retry = evaluate_explanation_json_only(
    case_results['TN']['explanation'],
    'TN'
)

tn_evaluation_summary_retry = safe_summarize_evaluation(
    tn_evaluation_text_retry,
    evaluation_weights
)

case_results['TN']['evaluation_text'] = tn_evaluation_text_retry
case_results['TN']['evaluation_summary'] = tn_evaluation_summary_retry

print(tn_evaluation_text_retry)
print('\nTN retry summary:')
print(tn_evaluation_summary_retry)

{
  "scores": {
    "faithfulness_no_hallucination": {
      "score": 3,
      "rationale": "The explanation mostly reflects the evidence but includes unsupported claims about the clinical significance of features and does not fully adhere to SHAP directions."
    },
    "clinical_plausibility": {
      "score": 4,
      "rationale": "The explanation is generally plausible and interprets most signals reasonably, though it lacks depth in clinical interpretation."
    },
    "caution_awareness": {
      "score": 5,
      "rationale": "Correctly mentions and explains relevant caution flags, such as the negative pre-ICU length of stay and non-clinical ICU unit identifier."
    },
    "completeness": {
      "score": 4,
      "rationale": "Covers most important risk-increasing and risk-decreasing factors with a clear overall interpretation, but could provide more detail on some aspects."
    },
    "clarity": {
      "score": 5,
      "rationale": "The explanation is very clear, concise, an

In [65]:
all_case_summary_df = pd.DataFrame([
    {
        'case_type': case_type,
        **result['evaluation_summary'],
        'forbidden_true_label_phrases': result['forbidden_true_label_phrases']
    }
    for case_type, result in case_results.items()
])

all_case_summary_df

,case_type,weighted_score,revision_needed,faithfulness_no_hallucination,clinical_plausibility,caution_awareness,completeness,clarity,overall_feedback,revision_instructions,parse_error,forbidden_true_label_phrases
0,FN,3.8,True,3,4,4,4,5,The explanation is mostly grounded in the evid...,Remove unsupported claims about 'normal' value...,None,[]
1,FP,3.8,True,3,4,4,4,5,The explanation is mostly faithful to the evid...,Ensure all statements are fully supported by t...,None,[]
2,TN,4.0,True,3,4,5,4,5,The explanation is mostly grounded in the evid...,Ensure all claims are supported by the evidenc...,None,[]


In [66]:
tp_agentic_results = {
    'initial_generated': {
        'explanation': tp_generated_explanation,
        'evaluation_text': tp_eval_text,
        'evaluation_summary': tp_initial_eval_json,
        'weighted_score': weighted_scores['initial_generated'],
        'forbidden_true_label_phrases': None
    },
    'first_revision': {
        'explanation': tp_revised_explanation,
        'evaluation_text': tp_revised_eval_text,
        'evaluation_summary': tp_revised_eval_json,
        'weighted_score': weighted_scores['first_revision'],
        'forbidden_true_label_phrases': None
    },
    'strict_second_revision': {
        'explanation': tp_strict_revised_explanation,
        'evaluation_text': tp_strict_revised_eval_text,
        'evaluation_summary': tp_strict_revised_eval_json,
        'weighted_score': weighted_scores['strict_second_revision'],
        'forbidden_true_label_phrases': strict_revision_forbidden_phrases
    }
}

print(tp_agentic_results.keys())

dict_keys(['initial_generated', 'first_revision', 'strict_second_revision'])


In [67]:
tp_agentic_summary_df = pd.DataFrame([
    {
        'version': version,
        'weighted_score': result['weighted_score'],
        'revision_needed': result['evaluation_summary']['revision_needed'],
        'faithfulness_no_hallucination': result['evaluation_summary']['scores']['faithfulness_no_hallucination']['score'],
        'clinical_plausibility': result['evaluation_summary']['scores']['clinical_plausibility']['score'],
        'caution_awareness': result['evaluation_summary']['scores']['caution_awareness']['score'],
        'completeness': result['evaluation_summary']['scores']['completeness']['score'],
        'clarity': result['evaluation_summary']['scores']['clarity']['score'],
        'forbidden_true_label_phrases': result['forbidden_true_label_phrases']
    }
    for version, result in tp_agentic_results.items()
])

tp_agentic_summary_df

,version,weighted_score,revision_needed,faithfulness_no_hallucination,clinical_plausibility,caution_awareness,completeness,clarity,forbidden_true_label_phrases
0,initial_generated,3.8,True,3,4,4,4,5,None
1,first_revision,4.0,True,3,4,5,4,5,None
2,strict_second_revision,4.7,True,4,5,5,5,5,[]


In [69]:
tp_final_summary = tp_agentic_results['strict_second_revision']
tp_final_eval = tp_final_summary['evaluation_summary']

final_summary_df = pd.DataFrame([
    {
        'case_type': 'TP',
        'weighted_score': tp_final_summary['weighted_score'],
        'revision_needed': tp_final_eval['revision_needed'],
        'faithfulness_no_hallucination': tp_final_eval['scores']['faithfulness_no_hallucination']['score'],
        'clinical_plausibility': tp_final_eval['scores']['clinical_plausibility']['score'],
        'caution_awareness': tp_final_eval['scores']['caution_awareness']['score'],
        'completeness': tp_final_eval['scores']['completeness']['score'],
        'clarity': tp_final_eval['scores']['clarity']['score'],
        'revision_rounds': 3,
        'parse_error': None,
        'forbidden_true_label_phrases': tp_final_summary['forbidden_true_label_phrases']
    }
] + [
    {
        'case_type': case_type,
        'weighted_score': result['evaluation_summary']['weighted_score'],
        'revision_needed': result['evaluation_summary']['revision_needed'],
        'faithfulness_no_hallucination': result['evaluation_summary']['faithfulness_no_hallucination'],
        'clinical_plausibility': result['evaluation_summary']['clinical_plausibility'],
        'caution_awareness': result['evaluation_summary']['caution_awareness'],
        'completeness': result['evaluation_summary']['completeness'],
        'clarity': result['evaluation_summary']['clarity'],
        'revision_rounds': 1,
        'parse_error': result['evaluation_summary']['parse_error'],
        'forbidden_true_label_phrases': result['forbidden_true_label_phrases']
    }
    for case_type, result in case_results.items()
])

final_summary_df

,case_type,weighted_score,revision_needed,faithfulness_no_hallucination,clinical_plausibility,caution_awareness,completeness,clarity,revision_rounds,parse_error,forbidden_true_label_phrases
0,TP,4.7,True,4,5,5,5,5,3,None,[]
1,FN,3.8,True,3,4,4,4,5,1,None,[]
2,FP,3.8,True,3,4,4,4,5,1,None,[]
3,TN,4.0,True,3,4,5,4,5,1,None,[]


In [70]:
output_dir = Path('../reports/llm_generation')
explanation_dir = output_dir / 'explanations'
evaluation_dir_out = output_dir / 'evaluations'

output_dir.mkdir(parents=True, exist_ok=True)
explanation_dir.mkdir(parents=True, exist_ok=True)
evaluation_dir_out.mkdir(parents=True, exist_ok=True)

tp_agentic_summary_df.to_csv(output_dir / 'tp_agentic_summary.csv', index=False)
all_case_summary_df.to_csv(output_dir / 'fn_fp_tn_summary.csv', index=False)
final_summary_df.to_csv(output_dir / 'final_llm_generation_summary.csv', index=False)

print('Summary files saved.')

Summary files saved.


In [71]:
for version, result in tp_agentic_results.items():
    (explanation_dir / f'tp_{version}_explanation.txt').write_text(
        result['explanation']
    )

    (evaluation_dir_out / f'tp_{version}_evaluation.txt').write_text(
        result['evaluation_text']
    )

for case_type, result in case_results.items():
    case_lower = case_type.lower()

    (explanation_dir / f'{case_lower}_explanation.txt').write_text(
        result['explanation']
    )

    (evaluation_dir_out / f'{case_lower}_evaluation.txt').write_text(
        result['evaluation_text']
    )

print('Explanation and evaluation text files saved.')

Explanation and evaluation text files saved.


In [72]:
with open(output_dir / 'tp_agentic_results.json', 'w') as f:
    json.dump(tp_agentic_results, f, indent=2)

with open(output_dir / 'fn_fp_tn_case_results.json', 'w') as f:
    json.dump(case_results, f, indent=2)

print('JSON result files saved.')

JSON result files saved.
